## 08 Evaluation, Benchmarking, and Monitoring

#### Purpose
- Give the final, unbiased result of the claim and lapse models on the **out-of-time 2 (validation)** holdout, 2023 to August 2024.
- Benchmark every model against the two naive rules, with the same measures as notebooks 06 and 07.
- Look at the models as classifiers (confusion matrices, ROC curves) and as probability models (log loss, Brier, calibration).
- Check where the models hold and where they fail (segments, years), and propose how to monitor them in production.

#### How we judge the models
As in 06a and 06b, we look at the models in two ways:
1. **As classifiers** (the view from our course): confusion matrices with accuracy, precision, recall and F1 at fixed cut-offs (claims: the riskiest 5%, 10%, 20%; lapse: 10%, 20%, 30%; and the default 0.5), and ROC curves with their AUC.
2. **As probability models:** log loss, Brier score, skill over the base rate, and calibration (predicted vs actual). The models are used for expected claims, expected lapses and pricing, so the probabilities themselves must be right.

#### Rules for this notebook
1. **The holdout is opened once.** Every choice (features, settings, which model) was made on the training years and on out-of-time 1 (test, 2021–2022). Here the models are **frozen**: no refitting, no new settings, no new features.
2. **Dry run first.** The switch `EVAL_SPLIT` in Step 0.2 is `"test"` until 07a and 07b are final: the whole notebook then runs on OOT1 (which we have already used) and never reads a 2023–2024 row. We set it to `"valid"` once, for the final run.
3. **Every model the same way.** Every model saved in `data/processed/models/` is scored and compared with the same code, so the 07a/07b models join automatically.

#### Steps:
0. Settings and the holdout switch
1. The models and the evaluation data (time split, outcome check)
2. Score the holdout
3. Benchmark table: naive vs baseline vs advanced
4. Classification view: confusion matrices and ROC curves
5. Calibration (deciles, calibration error and slope, expected vs actual)
6. Calibration by segment (state, flood zone, new business, mandatory purchase, storm during the term)
7. Stability over time (2014 to the last evaluation year)
8. Cohort survival analysis for lapse (retention of new business, actual vs predicted)
9. Simulated monitoring (feature and score drift, actual vs expected, alert rules)
10. Appendix: extra checks (bootstrap of the gains, policy history and linkage)
11. Conclusions and save

#### Authors
Organized and prepared by Sebastian Carino, Ma. Loisa Celorico, Cyrill Agustin, John Navarro, and Justino Rotairo III.

- **Date Created:** 2026-11-07
- **Date Modified:** 2026-11-16

### Section 0. Load the libraries and working directory
Use the same setup cell as notebook 02 (repo root, data folders, reports folder), so the notebook runs on every teammate's machine.

#### 0.1 Imports and paths

In [ ]:
%load_ext autoreload
%autoreload 2

# Find the repo root from any folder, so `src` can be imported
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import json
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from matplotlib.patches import Patch
from matplotlib.ticker import PercentFormatter
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (brier_score_loss, confusion_matrix, f1_score, log_loss, precision_score,
                             recall_score, roc_auc_score, roc_curve)

from src.paths import DOCS, PROCESSED, REPORTS, ensure_dirs
from src.db import connect, scalar
from src.features import apply_fixed_rules
pd.options.display.float_format = "{:,.4g}".format     # 4 significant digits, so small rates stay readable

ensure_dirs()

#### 0.2 Settings and the holdout switch
`EVAL_SPLIT` decides which out-of-time period is evaluated:

| `EVAL_SPLIT` | Years evaluated | When |
|---|---|---|
| `"test"` | OOT1, 2021–2022 | Dry run: as often as we like, while 07a/07b are being built |
| `"valid"` | OOT2, 2023 – Aug 2024 | **Once**, after 07a and 07b are final |

In the dry run, the stability (Section 7), cohort (Section 8) and monitoring (Section 9) sections also stop at 2022, so no 2023–2024 row is read before the final run. Every saved file carries the name of the split (`eval_test_…` or `eval_valid_…`), so the dry run never overwrites the final results.

In [ ]:
# Step 0.2 Settings. Change EVAL_SPLIT to "valid" ONCE, after notebooks 07a and 07b are final.
EVAL_SPLIT = "test"
assert EVAL_SPLIT in ("test", "valid")
EVAL_NAME = {"test": "OOT1 (test, 2021-2022)", "valid": "OOT2 (validation, 2023 - Aug 2024)"}[EVAL_SPLIT]
LAST_YEAR = 2022 if EVAL_SPLIT == "test" else 2024     # later years stay closed in the dry run
OUT = f"eval_{EVAL_SPLIT}"                             # prefix of every saved file

SEED = 42
N_CHUNKS = 8               # the evaluation split is scored in 8 chunks of about 0.9M rows
N_BOOT = 200               # bootstrap samples in Section 10.1
STABILITY_ROWS = 200_000   # rows per year in Section 7
REFERENCE_ROWS = 100_000   # training rows per calendar quarter, the reference in Section 9
QUARTER_ROWS = 100_000     # rows per quarter in Section 9
COHORT_KEYS = 30_000       # new-business policies followed per cohort in Section 8
TRAIN_ROWS = 300_000       # training rows that set the cut-offs in Section 4

# The two targets: population flag, target column and the segment rule of the naive benchmark (06a, 06b)
TARGETS = {
    "claim": {"target": "CLAIM_IN_TERM", "pop": "in_pop_claim", "event": "claim", "events": "claims",
              "segment": {"seg_1": "CASE WHEN zone_group = 'UNKNOWN' THEN 'D' ELSE zone_group END",
                          "seg_2": "CAST(COALESCE(had_prior_claim, FALSE) AS INTEGER)"},
              "segment_names": ("zone_group", "had_prior_claim")},
    "lapse": {"target": "LAPSE", "pop": "in_pop_lapse", "event": "lapse", "events": "lapses",
              "segment": {"seg_1": "CAST(COALESCE(tenure_years >= 0 AND tenure_years < 1, FALSE) AS INTEGER)",
                          "seg_2": "CAST(COALESCE(mandatory_purchase, FALSE) AS INTEGER)"},
              "segment_names": ("new_business", "mandatory_purchase")},
}
COLORS = ["#2a6fb0", "#d9822b", "#7b5ea7", "#3c9a5f"]   # one colour per model in the charts
SPLIT_OF_YEAR = lambda y: "train" if y <= 2020 else ("test" if y <= 2022 else "valid")

MODEL_DIR = PROCESSED / "models"
PRED_DIR = PROCESSED / "predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)

con = connect(db_path=None, memory_limit="6GB")
con.execute(f"CREATE OR REPLACE VIEW feats AS SELECT * FROM read_parquet('{PROCESSED / 'features.parquet'}')")
print(f"evaluating on {EVAL_NAME}; years up to {LAST_YEAR} may be read; outputs saved as {OUT}_*")

### Section 1. The models and the evaluation data

#### 1.1 The model registry
Every file `data/processed/models/<claim|lapse>_*.joblib` is a candidate. Each holds the same three parts, written by notebooks 06 and 07: the fitted preprocessing (`preprocess`), the fitted model (`model`, with `predict_proba`) and its feature list (`features`). Its run record (`reports/<name>_run.json`) tells us what it was trained on.

The two **naive benchmarks** are rebuilt here from the training split exactly as in 06a and 06b:
- **base rate:** the training rate for every policy-year;
- **segment rate:** the training rate of each segment (claims: zone group × earlier claim; lapse: new business × mandatory purchase). A segment not seen in training gets the base rate.

The **primary model** of each target is the advanced model if one is saved (07a/07b), otherwise the baseline. Sections 4–9 show its results in the charts; the tables cover every model.

In [ ]:
# Step 1.1 Load every saved model, rebuild the naive benchmarks from the training split
plan = pd.read_csv(DOCS / "feature_plan.csv").set_index("feature")
KIND = plan["kind"].to_dict()                      # num / cat / flag, for every feature

registry, base_rate, PRIMARY = {}, {}, {}
for kind, spec in TARGETS.items():
    registry[kind] = {}
    for path in sorted(MODEL_DIR.glob(f"{kind}_*.joblib")):
        saved = joblib.load(path)
        run_file = REPORTS / f"{path.stem}_run.json"
        saved["run"] = json.loads(run_file.read_text()) if run_file.exists() else {}
        registry[kind][path.stem] = saved
    assert registry[kind], f"no saved {kind} model in {MODEL_DIR}: run notebook 06{'a' if kind == 'claim' else 'b'} first"
    advanced = [s for s in registry[kind] if "baseline" not in s]
    PRIMARY[kind] = advanced[-1] if advanced else next(iter(registry[kind]))

    # Naive benchmarks, learned on the training split only (same SQL as 06a/06b)
    base_rate[kind] = scalar(con, f"SELECT AVG(CAST({spec['target']} AS DOUBLE)) FROM feats WHERE {spec['pop']} AND split = 'train'")
    seg_sql = ", ".join(f"{expr} AS {col}" for col, expr in spec["segment"].items())
    con.execute(f"""CREATE OR REPLACE TEMP TABLE seg_{kind} AS
                    SELECT {seg_sql}, COUNT(*) AS train_rows, AVG(CAST({spec['target']} AS DOUBLE)) AS segment_rate
                    FROM feats WHERE {spec['pop']} AND split = 'train' GROUP BY ALL""")

# One table: what each model is, what it was trained on, with which settings
rows = []
for kind, models in registry.items():
    for stem, m in models.items():
        run, params = m["run"], m["run"].get("params", {})
        rows.append({"target": kind, "model": stem, "primary": stem == PRIMARY[kind], "type": type(m["model"]).__name__,
                     "features": len(m["features"]), "trained_on": run.get("train_split", "?"),
                     "settings": ", ".join(f"{k}={params[k]}" for k in ("alpha", "eta0") if k in params),
                     "epochs": run.get("epochs"), "saved_at": run.get("saved_at")})
display(pd.DataFrame(rows))
pd.DataFrame([{"target": k, "base_rate_train": base_rate[k]} for k in TARGETS]).style.format({"base_rate_train": "{:.4%}"})

#### 1.2 The time split
The same split as in 06a and 06b (Step 1.2 there): the models learned on the **development** years (2014–2020), their settings were chosen with **in-time** rolling folds inside those years, and they were compared on **out-of-time 1** (2021–2022). This notebook gives the result on **out-of-time 2** (2023 to August 2024), which no step has used.

In [ ]:
# Step 1.2 The time split (the figure saved by notebook 06a)
fig_path = REPORTS / "fig_claim_time_split.png"
if fig_path.exists():
    display(Image(filename=str(fig_path), width=800))
else:
    print("run Step 1.2 of notebook 06a to draw the time split")

#### 1.3 Are the outcomes fully observed?
A policy-year can only be judged if its whole term has been observed: a claim in the last month of a term that the data do not yet cover would be missed, and the model would look as if it over-predicts. We check that the last term in the evaluation split ends before the claims data do, and how far the storm tracks (IBTrACS, used for the storm segment in Section 6) reach.

In [ ]:
# Step 1.3 Outcome check: terms in the evaluation split, and how far each source reaches
terms = con.sql(f"""
    SELECT period_year, COUNT(*) AS policy_years, MIN(period_start) AS first_start, MAX(period_start) AS last_start,
           MAX(period_end) AS last_end, AVG(exposure) AS mean_exposure
    FROM feats WHERE split = '{EVAL_SPLIT}' AND in_pop_claim GROUP BY ALL ORDER BY period_year""").df()
display(terms)

claims_end = scalar(con, f"SELECT MAX(loss_date) FROM read_parquet('{PROCESSED / 'claim_links' / '*.parquet'}')")
storm_end = scalar(con, f"SELECT MAX(wind_date) FROM read_parquet('{PROCESSED / 'storm_wind.parquet'}')")
last_end = terms["last_end"].max()
print(f"last term ends {last_end}; claims data reach {claims_end}; storm tracks reach {storm_end}")
print("claim outcomes fully observed:", pd.Timestamp(claims_end) >= pd.Timestamp(last_end))
if pd.Timestamp(storm_end) < pd.Timestamp(last_end):
    print("note: storms after the end of the IBTrACS file are missing from the storm segment (Section 6)")

### Section 2. Score the evaluation split

#### 2.1 Score every row with every model
We read the evaluation split chunk by chunk (as in 06a/06b), apply the same fixed clean-up rules (`apply_fixed_rules`), and score each row with every model and both benchmarks. For each row we also keep the columns used later to cut the results: state, flood zone, new business, mandatory purchase, policy history (Section 7) and the **strongest storm wind at the property during the term**.

The storm wind comes from the IBTrACS wind table of notebook 05, but here we look **inside** the term (between its start and its end). That is information from the outcome period: we use it only to describe *where* the models are right or wrong, never as a feature.

In [ ]:
# Step 2.1 Score the evaluation split with every model, chunk by chunk
# Storm wind during the term, for the evaluation rows only (an evaluation segment, never a feature)
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE storm_term AS
    SELECT f.period_id, MAX(w.wind_kt) AS term_wind_kt
    FROM feats f
    JOIN read_parquet('{PROCESSED / 'storm_wind.parquet'}') w
      ON w.latitude = CAST(f.latitude AS DOUBLE) AND w.longitude = CAST(f.longitude AS DOUBLE)
     AND w.wind_date BETWEEN f.period_start AND f.period_end
    WHERE f.split = '{EVAL_SPLIT}'
    GROUP BY f.period_id""")

def prepare(df, features):
    """Fixed rules and the column types the pipelines were fitted on (as in 06a/06b)."""
    df = apply_fixed_rules(df)
    for f in features:
        if KIND.get(f) == "cat":
            s = df[f].astype("string")
            df[f] = s.astype(object).where(s.notna(), np.nan)
        else:
            df[f] = pd.to_numeric(df[f], errors="coerce").astype("float32")
    return df

def score_rows(kind, where, sample_rows=None, seed=SEED):
    """Score the population rows of one target that match `where` with every model and both benchmarks.
    sample_rows: a random sample of that size (the same rows for the same seed)."""
    spec, models = TARGETS[kind], registry[kind]
    needed = sorted(set().union(*(m["features"] for m in models.values())))
    seg_sql = ", ".join(f"{expr} AS {col}" for col, expr in spec["segment"].items())
    query = f"""
        SELECT f.period_id, f.policy_key, f.period_year, f.period_start, f.propertyState AS state,
               f.{spec['target']} AS y, {seg_sql},
               CASE WHEN f.zone_group = 'UNKNOWN' THEN 'D' ELSE f.zone_group END AS ev_zone_group,
               CAST(COALESCE(f.tenure_years >= 0 AND f.tenure_years < 1, FALSE) AS INTEGER) AS ev_new_business,
               CAST(COALESCE(f.mandatory_purchase, FALSE) AS INTEGER) AS ev_mandatory_purchase,
               CASE WHEN f.prev_gap_days IS NULL THEN 'first term on the key'
                    WHEN f.returned_after_lapse THEN 'returned after a gap'
                    ELSE 'continuous renewal' END AS history,
               {', '.join(f'f.{c}' for c in needed)}
        FROM feats f WHERE f.{spec['pop']} AND {where}"""
    if sample_rows:
        query = f"SELECT * FROM ({query}) USING SAMPLE reservoir({sample_rows} ROWS) REPEATABLE ({seed})"
    query = f"""
        SELECT q.*, COALESCE(s.segment_rate, {base_rate[kind]}) AS p_segment, st.term_wind_kt
        FROM ({query}) q
        LEFT JOIN seg_{kind} s USING (seg_1, seg_2)
        LEFT JOIN storm_term st USING (period_id)"""
    df = prepare(con.sql(query).df(), needed)
    out = df[["period_id", "policy_key", "period_year", "period_start", "state", "history", "y", "p_segment"]].copy()
    out["y"] = out["y"].astype("int8")
    out["zone_group"] = df["ev_zone_group"]
    out["new_business"] = df["ev_new_business"]
    out["mandatory_purchase"] = df["ev_mandatory_purchase"]
    out["storm_in_term"] = pd.cut(df["term_wind_kt"].fillna(0), [-1, 33.9, 63.9, 999],
                                  labels=["no storm wind", "tropical-storm wind (34-63 kt)", "hurricane wind (64+ kt)"])
    out["p_base"] = base_rate[kind]
    for stem, m in models.items():
        out[f"p_{stem}"] = m["model"].predict_proba(m["preprocess"].transform(df[m["features"]]))[:, 1]
    return out, df

t0 = time.time()
pred = {}
for kind in TARGETS:
    parts = [score_rows(kind, f"f.split = '{EVAL_SPLIT}' AND hash(f.period_id || '{SEED}') % {N_CHUNKS} = {k}")[0]
             for k in range(N_CHUNKS)]
    pred[kind] = pd.concat(parts, ignore_index=True)
    print(f"{kind}: scored {len(pred[kind]):,} rows of {EVAL_NAME} ({(time.time() - t0) / 60:.1f} min so far); "
          f"actual rate {pred[kind]['y'].mean():.3%}")
del parts

# Probability columns, in the order shown in every table: the two benchmarks, then the models
MODELS = {kind: {"base rate": "p_base", "segment rate": "p_segment", **{stem: f"p_{stem}" for stem in registry[kind]}}
          for kind in TARGETS}

### Section 3. Benchmark table

#### 3.1 Naive vs baseline vs advanced
The same measures as 06a/06b, for the whole evaluation period and for each year. Let $y_i \in \{0,1\}$ be the outcome and $p_i$ the predicted probability.

$$\text{LogLoss} = -\frac{1}{n}\sum_{i=1}^{n}\Big[y_i\log p_i + (1-y_i)\log(1-p_i)\Big] \qquad \text{Brier} = \frac{1}{n}\sum_{i=1}^{n}(p_i - y_i)^2$$

$$\text{Skill} = 1 - \frac{\text{score}_{\text{model}}}{\text{score}_{\text{base rate}}} \qquad \text{A/E} = \frac{\sum_i y_i}{\sum_i p_i}$$

- **Log loss** (main metric) and **Brier score**: lower is better; both are proper scoring rules.
- **Skill**: the improvement over giving everyone the training rate, for the same rows. 0% = no better than the base rate.
- **A/E** (actual ÷ expected): 1 = the model expects as many events as happened; above 1 = it under-predicts.
- **ROC-AUC**: how well the model ranks policy-years (Section 4 shows the curves); 0.5 = no better than chance.

When `EVAL_SPLIT = "valid"`, the OOT1 results saved by 06a/06b (and 07a/07b) are shown underneath, so we can see whether the models hold up from the first to the second out-of-time period. In the dry run, Step 3.2 instead checks that the scores here match the ones saved by 06a/06b.

In [ ]:
# Step 3.1 Benchmark table: every model on the evaluation split, overall and per year
def prob_scores(y, p):
    y, p = np.asarray(y), np.clip(np.asarray(p, dtype="float64"), 1e-7, 1 - 1e-7)
    return {"rows": len(y), "events": int(y.sum()), "actual_rate": y.mean(), "predicted_rate": p.mean(),
            "a_to_e": y.sum() / p.sum(), "log_loss": log_loss(y, p, labels=[0, 1]), "brier": brier_score_loss(y, p)}

def add_skill(table, by):
    """Skill over the base-rate row of the same group (scope / segment)."""
    base = table[table["model"] == "base rate"].set_index(by)
    for m in ("log_loss", "brier"):
        table[f"{m}_skill"] = 1 - table[m] / table[by].map(base[m])
    return table

def benchmark(frame, models, scopes):
    rows = [{"scope": scope, "model": name, **prob_scores(part["y"], part[col]), "roc_auc": roc_auc_score(part["y"], part[col])}
            for scope, part in scopes for name, col in models.items()]
    return add_skill(pd.DataFrame(rows), "scope")

FMT = {"rows": "{:,}", "events": "{:,}", "actual_rate": "{:.3%}", "predicted_rate": "{:.3%}", "a_to_e": "{:.3f}",
       "log_loss": "{:.5f}", "brier": "{:.6f}", "log_loss_skill": "{:+.2%}", "brier_skill": "{:+.2%}", "roc_auc": "{:.4f}"}

results = {}
for kind in TARGETS:
    p = pred[kind]
    scopes = [(EVAL_NAME, p), *[(str(y), g) for y, g in p.groupby("period_year")]]
    results[kind] = benchmark(p, MODELS[kind], scopes)
    print(f"\n{kind.upper()}: {EVAL_NAME}")
    display(results[kind].style.format(FMT))

#### 3.2 Out-of-time 1 next to out-of-time 2 (or: dry-run check)
- **Final run (`"valid"`):** the OOT1 results from the saved test predictions of 06a/06b/07a/07b, with the same measures. A model whose skill falls sharply from OOT1 to OOT2 was tuned too closely to 2021–2022.
- **Dry run (`"test"`):** the evaluation split *is* OOT1, so the scores here must equal the ones saved by 06a/06b. A difference means a model was loaded or prepared differently from how it was trained.

In [ ]:
# Step 3.2 OOT1 results from the saved test predictions (06a/06b/07a/07b)
def oot1_table(kind):
    frames = {}
    for stem in registry[kind]:
        f = PRED_DIR / f"{stem}_test.parquet"
        if f.exists():
            t = pd.read_parquet(f)
            col = next(c for c in t.columns if c.startswith("p_") and c not in ("p_base", "p_segment"))
            frames[stem] = t.rename(columns={col: f"p_{stem}"})
    if not frames:
        return None
    t = next(iter(frames.values()))[["period_id", "y", "p_base", "p_segment"]]
    for stem, f in frames.items():
        t = t.merge(f[["period_id", f"p_{stem}"]], on="period_id", how="inner")
    models = {"base rate": "p_base", "segment rate": "p_segment", **{s: f"p_{s}" for s in frames}}
    return benchmark(t, models, [("OOT1 (saved test predictions)", t)])

for kind in TARGETS:
    saved = oot1_table(kind)
    if saved is None:
        print(f"{kind}: no saved test predictions")
        continue
    if EVAL_SPLIT == "test":
        here = results[kind][results[kind]["scope"] == EVAL_NAME].set_index("model")["log_loss"]
        check = saved.set_index("model")["log_loss"].to_frame("saved_by_06").join(here.rename("scored_here"))
        check["difference"] = check["scored_here"] - check["saved_by_06"]
        print(f"\n{kind.upper()}: dry-run check (the two columns should match)")
        display(check.style.format("{:.6f}"))
    else:
        both = pd.concat([saved, results[kind][results[kind]["scope"] == EVAL_NAME]], ignore_index=True)
        print(f"\n{kind.upper()}: out-of-time 1 vs out-of-time 2")
        display(both[["scope", "model", "actual_rate", "a_to_e", "log_loss", "brier", "log_loss_skill", "brier_skill", "roc_auc"]]
                .style.format(FMT))

### Section 4. Classification view: confusion matrices and ROC curves

#### 4.1 Confusion matrices at fixed cut-offs
A confusion matrix needs a cut-off *c*: flag the policy-year if $p_i \ge c$.

|  | Event ($y=1$) | No event ($y=0$) |
|---|---|---|
| **Flagged** ($p \ge c$) | TP | FP |
| **Not flagged** ($p < c$) | FN | TN |

$$\text{Accuracy} = \frac{TP+TN}{n} \qquad \text{Precision} = \frac{TP}{TP+FP} \qquad \text{Recall} = \frac{TP}{TP+FN} \qquad F_1 = \frac{2\cdot\text{Precision}\cdot\text{Recall}}{\text{Precision}+\text{Recall}}$$

We use the same rules as 06a/06b: flag the **riskiest 5%, 10% and 20%** of policy-years for claims and the **riskiest 10%, 20% and 30%** for lapse, plus the default **0.5** for comparison. Each cut-off is the matching quantile of the model's predictions on a sample of the **training** split, so the evaluation outcomes are never used to set it. A share flagged here that differs a lot from the rule's share means the scores have moved since training.

**Lift** (precision ÷ the actual rate) shows how many times more often the flagged group has the event than an average policy-year. With about 1% claims, a precision of a few percent is a lift of about 3, and the default 0.5 flags almost nobody: accuracy is then about 99% with a recall of 0, which is why we read precision, recall and F1 rather than accuracy.

In [ ]:
# Step 4.1 Confusion matrices for every model (except the base rate, which gives everyone the same score)
FLAG_SHARES = {"claim": [0.05, 0.10, 0.20], "lapse": [0.10, 0.20, 0.30]}

cm_rows = []
for kind in TARGETS:
    train_scored, _ = score_rows(kind, "f.split = 'train'", sample_rows=TRAIN_ROWS)     # sets the cut-offs
    y = pred[kind]["y"].to_numpy()
    for name, col in MODELS[kind].items():
        if name == "base rate":
            continue
        for share in [*FLAG_SHARES[kind], None]:
            cut = 0.5 if share is None else float(np.quantile(train_scored[col], 1 - share))
            flagged = (pred[kind][col].to_numpy() >= cut).astype(int)
            tn, fp, fn, tp = confusion_matrix(y, flagged, labels=[0, 1]).ravel()
            cm_rows.append({"target": kind, "model": name,
                            "rule": "p >= 0.5 (default)" if share is None else f"riskiest {share:.0%} in train",
                            "threshold": cut, "flagged_share": flagged.mean(), "TP": tp, "FP": fp, "FN": fn, "TN": tn,
                            "accuracy": (tp + tn) / len(y),
                            "precision": precision_score(y, flagged, zero_division=0),
                            "recall": recall_score(y, flagged, zero_division=0),
                            "f1": f1_score(y, flagged, zero_division=0),
                            "lift": precision_score(y, flagged, zero_division=0) / y.mean()})
cm_table = pd.DataFrame(cm_rows)

# The primary model at the middle rule, as a 2 x 2 table for each target
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, kind in zip(axes, TARGETS):
    t = cm_table[(cm_table["target"] == kind) & (cm_table["model"] == PRIMARY[kind])]
    r = t.iloc[1]
    m = np.array([[r["TN"], r["FP"]], [r["FN"], r["TP"]]])
    ax.imshow(m / m.sum(axis=1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
    for a in range(2):
        for b in range(2):
            share = m[a, b] / m[a].sum()
            ax.text(b, a, f"{m[a, b]:,}\n({share:.1%} of row)", ha="center", va="center", fontsize=10,
                    color="white" if share > 0.6 else "#333333")
    ev = TARGETS[kind]["event"]
    ax.set_xticks([0, 1], ["flagged: no", "flagged: yes"]); ax.set_yticks([0, 1], [f"actual: no {ev}", f"actual: {ev}"])
    ax.set_title(f"{kind.capitalize()}, {r['rule']}\nprecision {r['precision']:.1%}, recall {r['recall']:.1%}, "
                 f"accuracy {r['accuracy']:.1%}", fontsize=10)
plt.tight_layout()
plt.savefig(REPORTS / f"fig_{OUT}_confusion.png", dpi=150)
plt.show()
cm_table.style.format({"threshold": "{:.4f}", "flagged_share": "{:.1%}", "TP": "{:,}", "FP": "{:,}", "FN": "{:,}",
                       "TN": "{:,}", "accuracy": "{:.3f}", "precision": "{:.2%}", "recall": "{:.1%}", "f1": "{:.3f}",
                       "lift": "{:.2f}"})

#### 4.2 ROC curves
Every cut-off gives one confusion matrix; the ROC curve shows them all: the true positive rate (recall) against the false positive rate, $\text{FPR} = FP/(FP+TN)$, as the cut-off moves from 1 down to 0. The **AUC** is the probability that a random event gets a higher score than a random non-event (0.5 = chance, 1 = perfect). The dots mark the cut-off rules of the primary model from Step 4.1.

ROC-AUC judges the **ranking** only: a model can rank well and still expect twice too many events, which is what Sections 5 and 6 check.

In [ ]:
# Step 4.2 ROC curves of every model, one panel per target, with the cut-off rules of the primary model
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, kind in zip(axes, TARGETS):
    p = pred[kind]
    ax.plot([0, 1], [0, 1], color="#9a9a9a", linewidth=1, linestyle="--", label="random order (AUC 0.500)")
    for i, (name, col) in enumerate(n for n in MODELS[kind].items() if n[0] != "base rate"):
        fpr, tpr, _ = roc_curve(p["y"], p[col])
        style = dict(color="#9a9a9a", linewidth=1.5) if name == "segment rate" else dict(color=COLORS[(i - 1) % len(COLORS)], linewidth=2)
        ax.plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(p['y'], p[col]):.3f})", **style)
    t = cm_table[(cm_table["target"] == kind) & (cm_table["model"] == PRIMARY[kind]) & (cm_table["rule"] != "p >= 0.5 (default)")]
    x_r = t["FP"] / (t["FP"] + t["TN"])
    ax.scatter(x_r, t["recall"], color="#d9822b", s=50, zorder=3, edgecolor="white", label="cut-off rules (Step 4.1)")
    for xr, (_, r) in zip(x_r, t.iterrows()):
        ax.annotate(r["rule"].replace(" in train", ""), (xr, r["recall"]), textcoords="offset points", xytext=(8, -12), fontsize=8)
    ax.set(title=f"{kind.capitalize()}: ROC curves, {EVAL_NAME}", xlabel="false positive rate", ylabel="true positive rate (recall)",
           xlim=(0, 1), ylim=(0, 1))
    ax.legend(frameon=False, fontsize=8, loc="lower right")
    ax.spines[["top", "right"]].set_visible(False); ax.grid(color="#e6e6e6"); ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(REPORTS / f"fig_{OUT}_roc.png", dpi=150)
plt.show()

### Section 5. Calibration

#### 5.1 Calibration by decile, calibration error and slope
We sort the evaluation rows by each model's predicted probability, cut them into 10 equal groups (deciles) and compare the mean prediction $\bar p_d$ with the actual rate $\bar y_d$ in each.

**Expected calibration error (ECE):** the average gap, weighted by the size of each decile; we also show it as a share of the actual rate, so claims and lapse can be compared:
$$\text{ECE} = \sum_{d=1}^{10}\frac{n_d}{n}\,\big|\bar{y}_d - \bar{p}_d\big|$$

**Calibration intercept and slope:** we fit a logistic regression of the outcome on the model's own log-odds,
$$\log\frac{P(y_i=1)}{1-P(y_i=1)} = a + b\,\log\frac{p_i}{1-p_i}$$
- $b = 1$ and $a = 0$: perfectly calibrated.
- $b < 1$: the probabilities are **too extreme** (too high for the riskiest, too low for the safest), a sign of overfitting.
- $b > 1$: the probabilities are **too timid**; the model is right about the direction but shrinks the differences.
- $a \neq 0$ with $b \approx 1$: the overall level is off (see A/E), for example because the evaluation years had more storms.

We do not use the Hosmer–Lemeshow test: with millions of rows it rejects even a tiny, harmless miscalibration.

In [ ]:
# Step 5.1 Calibration by decile, ECE, and calibration intercept and slope (every model except the base rate)
def logit(p):
    p = np.clip(np.asarray(p, dtype="float64"), 1e-7, 1 - 1e-7)
    return np.log(p / (1 - p))

def calibration(frame, col, n_bins=10):
    d = pd.qcut(frame[col].rank(method="first"), n_bins, labels=range(1, n_bins + 1)).astype(int)
    t = frame.groupby(d).agg(rows=("y", "size"), events=("y", "sum"), predicted=(col, "mean"), actual=("y", "mean"))
    t["actual_to_predicted"] = t["actual"] / t["predicted"]
    return t

cal_rows, deciles = [], {}
for kind in TARGETS:
    p = pred[kind]
    sample = p.sample(min(len(p), 1_000_000), random_state=SEED)       # the slope fit needs no more rows than this
    for name, col in MODELS[kind].items():
        if name == "base rate":
            continue
        t = calibration(p, col)
        deciles[(kind, name)] = t
        ece = float((t["rows"] / t["rows"].sum() * (t["actual"] - t["predicted"]).abs()).sum())
        fit = LogisticRegression(penalty=None).fit(logit(sample[col]).reshape(-1, 1), sample["y"])
        cal_rows.append({"target": kind, "model": name, "a_to_e": p["y"].sum() / p[col].sum(),
                         "ece": ece, "ece_pct_of_rate": ece / p["y"].mean(),
                         "intercept_a": fit.intercept_[0], "slope_b": fit.coef_[0, 0]})
calib = pd.DataFrame(cal_rows)

# Chart: calibration by decile, one panel per target (segment rate in grey, models in colour)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for ax, kind in zip(axes, TARGETS):
    names = [n for n in MODELS[kind] if n != "base rate"]
    lim = max(max(deciles[(kind, n)]["predicted"].max(), deciles[(kind, n)]["actual"].max()) for n in names) * 1.08
    ax.plot([0, lim], [0, lim], color="#9a9a9a", linewidth=1, linestyle="--", label="perfect calibration")
    for i, n in enumerate(names):
        t = deciles[(kind, n)]
        style = dict(color="#9a9a9a", linewidth=1.5, marker="s", markersize=5) if n == "segment rate" else \
                dict(color=COLORS[(i - 1) % len(COLORS)], linewidth=2, marker="o", markersize=7, markeredgecolor="white")
        ax.plot(t["predicted"], t["actual"], label=n, **style)
    ax.set(title=f"{kind.capitalize()}: calibration by decile, {EVAL_NAME}", xlabel="mean predicted probability",
           ylabel="actual rate", xlim=(0, lim), ylim=(0, lim))
    ax.xaxis.set_major_formatter(PercentFormatter(1, decimals=1)); ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=1))
    ax.legend(frameon=False, fontsize=8)
    ax.spines[["top", "right"]].set_visible(False); ax.grid(color="#e6e6e6"); ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(REPORTS / f"fig_{OUT}_calibration.png", dpi=150)
plt.show()

for kind in TARGETS:
    print(f"{kind.upper()}: deciles of the primary model ({PRIMARY[kind]})")
    display(deciles[(kind, PRIMARY[kind])].sort_index(ascending=False).style.format(
        {"rows": "{:,}", "events": "{:,}", "predicted": "{:.3%}", "actual": "{:.3%}", "actual_to_predicted": "{:.2f}"}))
calib.style.format({"a_to_e": "{:.3f}", "ece": "{:.5f}", "ece_pct_of_rate": "{:.1%}", "intercept_a": "{:+.3f}", "slope_b": "{:.3f}"})

#### 5.2 Expected vs actual events
The sum of the probabilities is the number of events the model expects. This is how a pricing or reserving team would use it, so it is the most direct business check of the probabilities:
$$E[\text{events}] = \sum_i p_i \quad \text{vs.} \quad \sum_i y_i$$

In [ ]:
# Step 5.2 Expected (sum of probabilities) vs actual events, per year and model
expected = {}
for kind in TARGETS:
    p = pred[kind]
    t = p.groupby("period_year").agg(actual=("y", "sum"), **{f"expected: {n}": (c, "sum") for n, c in MODELS[kind].items()})
    t.loc["all"] = t.sum()
    expected[kind] = t
    print(f"{kind.upper()} ({EVAL_NAME})")
    display(t.style.format("{:,.0f}"))

### Section 6. Calibration by segment

#### 6.1 By state, flood zone, new business, mandatory purchase and storm during the term
A model can be right on average and wrong for whole groups: too low in one state and too high in another. For each segment we show the actual rate, the **A/E** of each model and the log-loss skill over the base rate of the same segment.

The chart shows the A/E of the primary model with a 95% interval from the number of events in the segment (a Poisson approximation, so the interval is wide when a segment has few events):
$$\text{A/E} \times \Big(1 \pm \frac{1.96}{\sqrt{\text{actual events}}}\Big)$$
The shaded band is 0.8–1.2, the "watch" band of the monitoring rules in Section 9.

**Storm during the term** splits the rows by the strongest IBTrACS wind at the property during the term (Section 2.1). We expect the claim model to **under-predict where a storm hit and over-predict where none did**: no model known at the start of a term can know where the next storm will land. What matters is that the two balance out over a portfolio, and that the model is not wrong in the same direction everywhere.

In [ ]:
# Step 6.1 A/E and skill by segment, for every model; chart for the primary model
def top_states(frame, n=10):
    keep = frame["state"].value_counts().index[:n]
    return frame["state"].where(frame["state"].isin(keep), "other states")

SEGMENTS = {"state (largest 10)": lambda f: top_states(f), "flood zone group": lambda f: f["zone_group"],
            "new business": lambda f: f["new_business"].map({1: "new business", 0: "renewal"}),
            "mandatory purchase": lambda f: f["mandatory_purchase"].map({1: "mandatory", 0: "voluntary"}),
            "storm during the term": lambda f: f["storm_in_term"]}

def segment_table(frame, models, key):
    g = key(frame).astype("string").fillna("(missing)")
    rows = [{"segment": s, "model": name, **prob_scores(part["y"], part[col])}
            for s, part in frame.groupby(g, observed=True) for name, col in models.items()]
    return add_skill(pd.DataFrame(rows), "segment")

seg_results = {}
for kind in TARGETS:
    for label, key in SEGMENTS.items():
        seg_results[(kind, label)] = segment_table(pred[kind], MODELS[kind], key)

# Chart: A/E of the primary model by segment, with a 95% interval
fig, axes = plt.subplots(1, 2, figsize=(12, 9), sharey=True)
for ax, kind in zip(axes, TARGETS):
    labels, ae, lo, hi, ypos, y0, ticks = [], [], [], [], [], 0, []
    for label in SEGMENTS:
        t = seg_results[(kind, label)]
        t = t[t["model"] == PRIMARY[kind]]
        ticks.append((y0, label))
        for _, r in t.iterrows():
            half = 1.96 / np.sqrt(max(r["events"], 1))
            labels.append(f"{r['segment']}"); ae.append(r["a_to_e"]); lo.append(r["a_to_e"] * (1 - half)); hi.append(r["a_to_e"] * (1 + half))
            ypos.append(y0); y0 += 1
        y0 += 1                                                      # a gap between segment types
    ax.axvspan(0.8, 1.2, color="#eef3f9", zorder=0)
    ax.axvline(1, color="#555555", linewidth=1)
    ax.hlines(ypos, lo, hi, color="#2a6fb0", linewidth=2)
    ax.plot(ae, ypos, "o", color="#2a6fb0", markersize=7, markeredgecolor="white")
    ax.set_yticks(ypos, labels, fontsize=8)
    for y, label in ticks:
        ax.annotate(label, xy=(0, y - 0.9), xycoords=("axes fraction", "data"), fontsize=8, fontweight="bold", color="#555555")
    ax.set(title=f"{kind.capitalize()}: actual / expected by segment\n{PRIMARY[kind]}, {EVAL_NAME}", xlabel="A/E (1 = right on average; band = 0.8-1.2)")
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="x", color="#e6e6e6"); ax.set_axisbelow(True)
axes[0].invert_yaxis()
plt.tight_layout()
plt.savefig(REPORTS / f"fig_{OUT}_segments.png", dpi=150)
plt.show()

for (kind, label), t in seg_results.items():
    print(f"{kind.upper()}: {label}")
    display(t[["segment", "model", "rows", "events", "actual_rate", "a_to_e", "log_loss_skill"]].style.format(FMT))

### Section 7. Stability over time

#### 7.1 Every year from 2014 to the last evaluation year
We score a random sample of every year with the frozen models and follow the A/E and the log-loss skill from year to year:

| Years | What it shows |
|---|---|
| 2014–2020 | **In-sample** (the development years the models learned from): how well the models fit, including the storm years 2016–2017 |
| 2021–2022 | **Out-of-time 1**: the years used to compare models |
| 2023–Aug 2024 | **Out-of-time 2**: the final holdout (final run only) |

Claims are driven by storms, so we expect the claim A/E to swing from year to year even for a good model: high in a big storm year (more claims than an average year), low in a calm one. What we look for is a **trend** (A/E drifting steadily away from 1) or skill falling below 0 in the out-of-time years.

In [ ]:
# Step 7.1 A/E and log-loss skill per year, 2014 to LAST_YEAR, on a sample of each year
years = list(range(2014, LAST_YEAR + 1))
stab_rows = []
t0 = time.time()
for kind in TARGETS:
    for y in years:
        s, _ = score_rows(kind, f"f.split = '{SPLIT_OF_YEAR(y)}' AND f.period_year = {y}", sample_rows=STABILITY_ROWS)
        for name, col in MODELS[kind].items():
            stab_rows.append({"target": kind, "year": y, "period": {"train": "in-sample", "test": "OOT1", "valid": "OOT2"}[SPLIT_OF_YEAR(y)],
                              "model": name, **prob_scores(s["y"], s[col])})
    print(f"{kind}: {len(years)} years scored ({(time.time() - t0) / 60:.1f} min so far)")
stability = pd.DataFrame(stab_rows)
base = (stability[stability["model"] == "base rate"][["target", "year", "log_loss", "brier"]]
        .rename(columns={"log_loss": "log_loss_base", "brier": "brier_base"}))
stability = stability.merge(base, on=["target", "year"])
for m in ("log_loss", "brier"):
    stability[f"{m}_skill"] = 1 - stability[m] / stability[f"{m}_base"]

# Chart: A/E (top) and log-loss skill (bottom) per year, one column per target
PERIOD_SHADE = {"in-sample": "#f4f4f4", "OOT1": "#fbeee0", "OOT2": "#efeaf6"}
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
for j, kind in enumerate(TARGETS):
    t = stability[stability["target"] == kind]
    names = [n for n in MODELS[kind] if n != "base rate"]
    for i, (metric, ylabel) in enumerate([("a_to_e", "A/E (actual / expected)"), ("log_loss_skill", "log-loss skill over base rate")]):
        ax = axes[i, j]
        for period, color in PERIOD_SHADE.items():
            yrs = t.loc[t["period"] == period, "year"]
            if len(yrs):
                ax.axvspan(yrs.min() - 0.5, yrs.max() + 0.5, color=color, zorder=0)
        for k, n in enumerate(names):
            s = t[t["model"] == n]
            style = dict(color="#9a9a9a", linewidth=1.5, marker="s", markersize=5) if n == "segment rate" else \
                    dict(color=COLORS[(k - 1) % len(COLORS)], linewidth=2, marker="o", markersize=6, markeredgecolor="white")
            ax.plot(s["year"], s[metric], label=n, **style)
        ax.axhline(1 if metric == "a_to_e" else 0, color="#555555", linewidth=1)
        ax.set(ylabel=ylabel, title=f"{kind.capitalize()}: {ylabel}" if i == 0 else None)
        if metric == "log_loss_skill":
            ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
        ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", color="#e6e6e6"); ax.set_axisbelow(True)
    axes[1, j].set(xlabel="policy year (2024: January to August)")
    axes[1, j].set_xticks(years)
axes[0, 0].legend(handles=[*axes[0, 0].get_legend_handles_labels()[0],
                           *[Patch(color=c, label=p) for p, c in PERIOD_SHADE.items() if p in set(stability["period"])]],
                  frameon=False, fontsize=8, loc="upper left")
plt.tight_layout()
plt.savefig(REPORTS / f"fig_{OUT}_stability.png", dpi=150)
plt.show()
stability.pivot_table(index=["target", "year", "period"], columns="model", values=["a_to_e", "log_loss_skill"]).round(3)

### Section 8. Cohort survival analysis for lapse
Sections 3–7 judge the lapse model one policy-year at a time. An insurer also needs to know **how long a new policy stays**: of the policies sold in a year, how many are still in force after 1, 2, 3, … years? That is a **survival** (retention) curve, the insurance version of a cohort or flow-rate analysis.

#### 8.1 Cohorts of new business: actual vs predicted retention
**Cohort:** all policies that start as **new business** (tenure under one year, with no earlier policy-year on the same key) in the same calendar year: the 2014 cohort, the 2015 cohort, and so on. We follow a random sample of 30,000 policies per cohort through their later policy-years. Year $k$ is the $k$-th policy-year since the policy was sold.

**Flow table (life table).** In each year $k$, policies at risk either lapse (flow out: a cancellation during the term or a non-renewal), renew into year $k+1$, or are **censored**: we stop seeing them without a lapse, because the data end (a 2021 policy can only be followed to 2022 in the dry run) or because a later policy-year left the modeling population. A censored policy is not counted as a lapse.
$$h_k = \frac{\text{lapsed in year } k}{\text{at risk at the start of year } k} \qquad S(k) = \prod_{j=1}^{k}\big(1 - h_j\big)$$
$h_k$ is the **lapse hazard** (the flow rate out) and $S(k)$ the **survival**, the share of the cohort still in force after $k$ years. This is the Kaplan–Meier (life-table) estimator for yearly steps.

**What the model predicts.** Our lapse model is a logistic regression on policy-years, which is a **discrete-time hazard model**: each prediction $p_{ik}$ is the probability that policy $i$ lapses in year $k$, given that it is still in force. Averaging over the same policies at risk gives the model's hazard and survival, directly comparable with the actual curve:
$$\hat h_k = \frac{1}{n_k}\sum_{i \,\in\, \text{at risk at } k} p_{ik} \qquad \hat S(k) = \prod_{j=1}^{k}\big(1 - \hat h_j\big)$$
Because both curves use the same policies at risk, censoring affects them in the same way. A gap between $S(k)$ and $\hat S(k)$ that **grows with $k$** means the model's errors add up over the years, which matters for premium-income forecasts several years ahead. We show the primary model and the segment rule (new business × mandatory purchase).

**Limits:**
- Only cohorts sold from 2014 onward: policies already in force in 2014 were sold earlier, and we do not see their first years (left truncation).
- Later cohorts have shorter curves; we show a year only while at least 500 policies of the cohort are at risk.
- A policy that lapses and later comes back starts a new spell; we follow only the first spell, up to the first lapse.
- The years after 2020 are out-of-time for the model; in the dry run nothing after 2022 is read.

In [ ]:
# Step 8.1 New-business cohorts: follow a sample of policies year by year; actual vs predicted lapse hazard and survival
MIN_AT_RISK = 500
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE cohort_keys AS
    WITH entry AS (                      -- the first new-business policy-year of each key, 2014 to LAST_YEAR
        SELECT policy_key, MIN(period_start) AS entry_start
        FROM feats
        WHERE in_pop_lapse AND tenure_years >= 0 AND tenure_years < 1 AND period_year BETWEEN 2014 AND {LAST_YEAR}
        GROUP BY policy_key
    ), clean AS (                        -- no earlier policy-year on the key: really new, not a return after a lapse
        SELECT e.policy_key, e.entry_start, year(e.entry_start) AS cohort
        FROM entry e
        WHERE NOT EXISTS (SELECT 1 FROM feats f WHERE f.policy_key = e.policy_key AND f.period_start < e.entry_start)
    )
    SELECT * FROM clean
    QUALIFY row_number() OVER (PARTITION BY cohort ORDER BY hash(CAST(policy_key AS VARCHAR) || '{SEED}')) <= {COHORT_KEYS}""")
cohort_years = sorted(r[0] for r in con.execute("SELECT DISTINCT cohort FROM cohort_keys").fetchall())

t0 = time.time()
parts = []
for c in cohort_years:                   # one cohort at a time, every policy-year of its sampled policies
    s, _ = score_rows("lapse", f"f.period_year <= {LAST_YEAR} AND f.policy_key IN (SELECT policy_key FROM cohort_keys WHERE cohort = {c})")
    parts.append(s)
cohort_rows = pd.concat(parts, ignore_index=True)
del parts
extra = con.sql(f"""
    SELECT f.period_id, CAST(f.CANCEL_IN_TERM AS INTEGER) AS cancelled, CAST(f.NON_RENEWAL AS INTEGER) AS not_renewed
    FROM feats f JOIN cohort_keys k USING (policy_key)
    WHERE f.in_pop_lapse AND f.period_year <= {LAST_YEAR}""").df()
cohort_rows = (cohort_rows.merge(con.sql("SELECT * FROM cohort_keys").df(), on="policy_key")
                          .merge(extra, on="period_id", how="left"))

# Year since new business, and only the first spell (up to and including the first lapse)
cohort_rows = cohort_rows[cohort_rows["period_start"] >= cohort_rows["entry_start"]].sort_values(["policy_key", "period_start"])
cohort_rows["year_k"] = cohort_rows.groupby("policy_key").cumcount() + 1
lapsed_before = cohort_rows.groupby("policy_key")["y"].cumsum() - cohort_rows["y"]
cohort_rows = cohort_rows[lapsed_before == 0]
print(f"{cohort_rows['policy_key'].nunique():,} policies in {len(cohort_years)} cohorts, {len(cohort_rows):,} policy-years "
      f"({(time.time() - t0) / 60:.1f} min)")

CURVES = {"model": f"p_{PRIMARY['lapse']}", "segment rule": "p_segment"}

def life_table(frame):
    """Flow table by year since new business: at risk, lapsed (cancelled / not renewed), censored, hazard, survival."""
    t = frame.groupby("year_k").agg(at_risk=("y", "size"), lapsed=("y", "sum"), cancelled=("cancelled", "sum"),
                                    not_renewed=("not_renewed", "sum"), **{f"hazard_{n}": (c, "mean") for n, c in CURVES.items()})
    t = t[t["at_risk"] >= MIN_AT_RISK]
    t["censored"] = t["at_risk"] - t["lapsed"] - t["at_risk"].shift(-1, fill_value=0)
    t["hazard_actual"] = t["lapsed"] / t["at_risk"]
    for n in ["actual", *CURVES]:
        t[f"survival_{n}"] = (1 - t[f"hazard_{n}"]).cumprod()
    return t

tables = {c: life_table(g) for c, g in cohort_rows.groupby("cohort")}
tables["all cohorts"] = life_table(cohort_rows)
cohort_tables = pd.concat(tables, names=["cohort"]).reset_index()

# Check: survival actual vs predicted at years 1, 2, 3 and 5, per cohort (percentage points)
cohort_check = cohort_tables[cohort_tables["year_k"].isin([1, 2, 3, 5])].assign(
    gap_model_pp=lambda t: 100 * (t["survival_model"] - t["survival_actual"]),
    gap_segment_pp=lambda t: 100 * (t["survival_segment rule"] - t["survival_actual"]))[
    ["cohort", "year_k", "at_risk", "survival_actual", "survival_model", "survival_segment rule", "gap_model_pp", "gap_segment_pp"]]

# Chart 1: one panel per cohort, actual vs predicted survival
n = len(cohort_years)
ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3.2 * nrows), sharex=True, sharey=True, squeeze=False)
for ax, c in zip(axes.ravel(), cohort_years):
    t = tables[c]
    ax.plot(t.index, t["survival_actual"], color="#333333", linewidth=2, marker="o", markersize=5, label="actual (Kaplan-Meier)")
    ax.plot(t.index, t["survival_model"], color="#2a6fb0", linewidth=2, linestyle="--", marker="s", markersize=4, label=f"model ({PRIMARY['lapse']})")
    ax.plot(t.index, t["survival_segment rule"], color="#9a9a9a", linewidth=1.5, linestyle=":", label="segment rule")
    ax.set_title(f"{c} cohort ({t['at_risk'].iloc[0]:,} policies)", fontsize=10)
    ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
    ax.spines[["top", "right"]].set_visible(False); ax.grid(color="#e6e6e6"); ax.set_axisbelow(True)
for ax in axes.ravel()[n:]:
    ax.set_visible(False)
for ax in axes[-1]:
    ax.set_xlabel("years since new business")
for ax in axes[:, 0]:
    ax.set_ylabel("share still in force")
axes[0, 0].set_ylim(0, 1.02)
axes[0, 0].legend(frameon=False, fontsize=8, loc="lower left")
plt.tight_layout()
plt.savefig(REPORTS / f"fig_{OUT}_lapse_cohort_survival.png", dpi=150)
plt.show()

display(tables["all cohorts"][["at_risk", "lapsed", "cancelled", "not_renewed", "censored", "hazard_actual", "hazard_model",
                               "survival_actual", "survival_model", "survival_segment rule"]].style.format(
    {"at_risk": "{:,}", "lapsed": "{:,}", "cancelled": "{:,}", "not_renewed": "{:,}", "censored": "{:,}",
     "hazard_actual": "{:.2%}", "hazard_model": "{:.2%}", "survival_actual": "{:.1%}", "survival_model": "{:.1%}",
     "survival_segment rule": "{:.1%}"}))
cohort_check.style.format({"at_risk": "{:,}", "survival_actual": "{:.1%}", "survival_model": "{:.1%}",
                           "survival_segment rule": "{:.1%}", "gap_model_pp": "{:+.1f}", "gap_segment_pp": "{:+.1f}"})

#### 8.2 Retention by segment
The same curves for groups defined **at the time of sale**, pooling all cohorts: mandatory vs voluntary purchase (a lender's requirement should keep policies in force), and the flood zone group. A model that is right overall can still be too optimistic for one group and too pessimistic for another; the solid line is the actual retention, the dashed line the model's.

In [ ]:
# Step 8.2 Actual vs predicted retention by segment at the time of sale (all cohorts pooled)
at_sale = cohort_rows.sort_values("year_k").groupby("policy_key")[["mandatory_purchase", "zone_group"]].first()
seg_rows = cohort_rows.drop(columns=["mandatory_purchase", "zone_group"]).join(at_sale, on="policy_key")
SEG_AT_SALE = {"mandatory purchase at sale": seg_rows["mandatory_purchase"].map({1: "mandatory", 0: "voluntary"}),
               "flood zone group at sale": seg_rows["zone_group"].astype("string").fillna("(missing)")}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
seg_tables = {}
for ax, (label, key) in zip(axes, SEG_AT_SALE.items()):
    for i, (g, part) in enumerate(seg_rows.groupby(key)):
        t = life_table(part)
        if t.empty:
            continue
        seg_tables[(label, g)] = t
        color = COLORS[i % len(COLORS)]
        ax.plot(t.index, t["survival_actual"], color=color, linewidth=2, marker="o", markersize=4, label=f"{g}: actual")
        ax.plot(t.index, t["survival_model"], color=color, linewidth=1.5, linestyle="--", label=f"{g}: model")
    ax.set(title=label, xlabel="years since new business", ylim=(0, 1.02))
    ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
    ax.legend(frameon=False, fontsize=8, ncol=2)
    ax.spines[["top", "right"]].set_visible(False); ax.grid(color="#e6e6e6"); ax.set_axisbelow(True)
axes[0].set_ylabel("share still in force")
plt.tight_layout()
plt.savefig(REPORTS / f"fig_{OUT}_lapse_cohort_segments.png", dpi=150)
plt.show()
(pd.concat(seg_tables, names=["segment_type", "segment"])
   .reset_index()
   .query("year_k in [1, 2, 3, 5]")[["segment_type", "segment", "year_k", "at_risk", "survival_actual", "survival_model"]]
   .style.format({"at_risk": "{:,}", "survival_actual": "{:.1%}", "survival_model": "{:.1%}"}))

#### Our findings (Section 8)

**8.1 Retention of new business**

1. **A new flood policy has a short life.** Pooling the 2014–2022 cohorts, **24% of new policies lapse in their first year**, and only **64% are still in force after two years, 55% after three and 43% after five**. The lapse hazard is highest in the first year (23.8%), falls to 15.6% in the second and settles at about 10–12% a year from the fourth year. The first renewal is the main point of loss.
2. **Almost every lapse is a non-renewal.** In the first year, 64,284 of the 64,293 lapses were non-renewals and only 672 policies were cancelled during the term (some policies have both flags). Retention work should therefore focus on the **renewal date**, not on cancellations during the term.
3. **The model follows retention closely in the development years.** For the pooled cohorts, the predicted survival is within **1 percentage point** of the actual survival in each of the first five years (76.2% vs 75.8% after year 1, 43.2% vs 44.2% after year 5). The errors do not add up over the years, which matters for premium-income forecasts several years ahead. The segment rule is close on the pooled curve too, but it misses by more for individual cohorts: for the 2015 cohort it predicted 77.0% survival after year 1 against 70.5% actual, while the model predicted 73.1%.
4. **The gap opens in the out-of-time years, and it follows the calendar, not the age of the policy.** The model **over-predicts lapse in 2022**:
   - the 2022 cohort kept **79.5%** of its policies in year 1, while the model expected **70.7%** (hazard 20.5% vs 29.3%);
   - the 2021 cohort, in its second year (2022), lapsed at **11.0%** against **17.4%** predicted.

   Both are calendar year 2022, the first full year of **Risk Rating 2.0**: premiums and price features changed (Section 9), and the model reads the higher prices as a higher lapse risk, but fewer policyholders left than in the training years. The 2019 cohort shows the opposite in 2020–2021 (years 2–3): more lapses than expected, so the model's survival is 4.6 points too high after year 3.

**8.2 Retention by segment at sale**

5. **Mandatory purchase keeps policies in force**, as expected: 53.7% of mandatory-purchase policies are still in force after five years against 40.8% of voluntary ones. The model gets the voluntary group right (within 1 point every year) but is **slightly too optimistic for mandatory purchase from year 3 onward** (56.6% vs 53.7% after year 5), probably because some owners pay off or refinance their mortgage and the requirement ends.
6. **The flood zone at sale shows the clearest bias.**
   - **A and V zones:** the model is **too pessimistic**. It expects 77.7% retention after year 1 in zone A, against 81.8% actual, and the gap stays at about 4–5 points for five years.
   - **Zone X** (outside the high-risk area): it is **too optimistic**, 74.4% predicted vs 71.9% actual after year 1, and the gap grows to 4 points after year 5.

   The model under-uses the zone. Owners in the high-risk zones keep their cover longer than the model's features suggest, and owners outside them drop it sooner. Zone D has too few policies (271) to read.

**Conclusion.** As a hazard model, the lapse baseline gives realistic multi-year retention curves for the portfolio in the development years. Its errors come from **segments** (flood zone, mandatory purchase) and from the **2022 price change**, not from the length of the forecast. Before using it for premium-income forecasts, we would (a) let the advanced model (07b) learn interactions between zone and tenure, and (b) recalibrate it on post-Risk Rating 2.0 data, and confirm on the 2023–2024 holdout whether the 2022 over-prediction persists.

### Section 9. Simulated monitoring
Treat each quarter after the development years as if it had just arrived in production, and check what a monitoring team would see. The **reference** is a random sample of the **same calendar quarter** in the development years (2014–2020), the data the models learned from. We compare like with like because several features are seasonal (start month, rain in the last 90 days, days since the last storm): a first quarter compared with a whole year would always look shifted.

#### 9.1 Feature drift and score drift (PSI)
The **population stability index** measures how much a distribution has moved from the reference. Each feature is cut into bins on the reference (10 quantile bins for numbers, the 15 largest categories plus "other" for categories, and missing as a bin of its own); with $r_b$ the reference share and $c_b$ the current share in bin $b$:
$$\text{PSI} = \sum_{b}\big(c_b - r_b\big)\,\ln\frac{c_b}{r_b}$$

| PSI | Reading (usual rule of thumb) |
|---|---|
| below 0.10 | stable |
| 0.10 – 0.25 | moderate shift: **watch** |
| above 0.25 | large shift: **review** the feature (a data change or a real change in the book) |

The same index on the predicted probability (**score PSI**) shows whether the mix of risks the model sees has changed. A large feature PSI is not an error by itself. **Risk Rating 2.0** (from October 2021) changed the premiums, the deductible options and the rate-type codes (`subsidized_rate_type`), so we expect the price and deductible features to cross 0.25 in every quarter after 2021. That is a real change in the book, not a data error. Features that measure time (days since the last storm or disaster, years since purchase, storms per year) also drift by construction as the years pass. Drift matters when the A/E moves too (Step 9.2).

In [ ]:
# Step 9.1 PSI of every feature of the primary model, and of its score, per quarter after 2020
def psi(ref, cur, numeric, n_bins=10, top=15, eps=1e-4):
    ref, cur = pd.Series(ref), pd.Series(cur)
    if numeric:
        r = pd.to_numeric(ref, errors="coerce").astype("float64"); c = pd.to_numeric(cur, errors="coerce").astype("float64")
        edges = np.unique(np.nanquantile(r, np.linspace(0, 1, n_bins + 1))[1:-1]) if r.notna().any() else np.array([])
        rb = pd.Series(np.where(r.isna(), -1, np.digitize(r.fillna(0), edges)))
        cb = pd.Series(np.where(c.isna(), -1, np.digitize(c.fillna(0), edges)))
    else:
        r = ref.astype("string").fillna("(missing)"); c = cur.astype("string").fillna("(missing)")
        keep = r.value_counts().index[:top]
        rb, cb = r.where(r.isin(keep), "other"), c.where(c.isin(keep), "other")
    p, q = rb.value_counts(normalize=True), cb.value_counts(normalize=True)
    p, q = p.reindex(p.index.union(q.index), fill_value=0) + eps, q.reindex(p.index.union(q.index), fill_value=0) + eps
    return float(((q - p) * np.log(q / p)).sum())

quarters = [(y, q) for y in range(2021, LAST_YEAR + 1) for q in range(1, 5) if not (y == 2024 and q == 4)]
monitor, psi_tables = [], {}
t0 = time.time()
for kind in TARGETS:
    stem, col = PRIMARY[kind], f"p_{PRIMARY[kind]}"
    feats_used = registry[kind][stem]["features"]
    # Reference: the same calendar quarter of 2014-2020 (several features are seasonal)
    refs = {q: score_rows(kind, f"f.split = 'train' AND quarter(f.period_start) = {q}", sample_rows=REFERENCE_ROWS)
            for q in range(1, 5)}
    psi_rows = {}
    for y, q in quarters:
        ref_scored, ref_raw = refs[q]
        where = f"f.split = '{SPLIT_OF_YEAR(y)}' AND year(f.period_start) = {y} AND quarter(f.period_start) = {q}"
        cur_scored, cur_raw = score_rows(kind, where, sample_rows=QUARTER_ROWS)
        label = f"{y}Q{q}"
        psi_rows[label] = {f: psi(ref_raw[f], cur_raw[f], numeric=KIND.get(f) == "num") for f in feats_used}
        monitor.append({"target": kind, "quarter": label, "rows": len(cur_scored),
                        "score_psi": psi(ref_scored[col], cur_scored[col], numeric=True),
                        "max_feature_psi": max(psi_rows[label].values()),
                        "features_over_0.25": sum(v > 0.25 for v in psi_rows[label].values()),
                        **{k: v for k, v in prob_scores(cur_scored["y"], cur_scored[col]).items() if k in ("actual_rate", "predicted_rate", "a_to_e", "log_loss")},
                        "log_loss_base": prob_scores(cur_scored["y"], cur_scored["p_base"])["log_loss"]})
    psi_tables[kind] = pd.DataFrame(psi_rows)
    print(f"{kind}: reference + {len(quarters)} quarters scored ({(time.time() - t0) / 60:.1f} min so far)")
monitor = pd.DataFrame(monitor)
monitor["log_loss_skill"] = 1 - monitor["log_loss"] / monitor["log_loss_base"]

# Heatmap: the 15 features of each primary model with the largest PSI in any quarter
fig, axes = plt.subplots(1, 2, figsize=(17, 6.5), gridspec_kw={"wspace": 0.6})
cmap = plt.get_cmap("Blues")
for ax, kind in zip(axes, TARGETS):
    t = psi_tables[kind]
    t = t.loc[t.max(axis=1).sort_values(ascending=False).index[:15]]
    im = ax.imshow(t.to_numpy(), cmap=cmap, vmin=0, vmax=0.5, aspect="auto")
    ax.set_xticks(range(t.shape[1]), t.columns, rotation=90, fontsize=8)
    ax.set_yticks(range(t.shape[0]), t.index, fontsize=8)
    for i in range(t.shape[0]):
        for j in range(t.shape[1]):
            v = t.iat[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6.5, color="white" if v > 0.3 else "#333333")
    ax.set_title(f"{kind.capitalize()}: feature PSI vs 2014-2020 (largest 15)")
fig.colorbar(im, ax=axes, shrink=0.7, label="PSI (0.10 watch, 0.25 review)")
plt.savefig(REPORTS / f"fig_{OUT}_psi.png", dpi=150, bbox_inches="tight")
plt.show()
monitor.style.format({"rows": "{:,}", "score_psi": "{:.3f}", "max_feature_psi": "{:.3f}", "actual_rate": "{:.3%}",
                      "predicted_rate": "{:.3%}", "a_to_e": "{:.3f}", "log_loss": "{:.5f}", "log_loss_base": "{:.5f}",
                      "log_loss_skill": "{:+.2%}"})

#### 9.2 Actual vs expected per quarter, and the alert rules
Drift in the inputs only matters if the probabilities go wrong, so the main monitoring measure is the **A/E per quarter** of the primary model, together with its log-loss skill. We propose these rules:

| Check | OK | Watch | Action |
|---|---|---|---|
| Feature PSI | < 0.10 | 0.10 – 0.25 | > 0.25: review the feature's data and definition |
| Score PSI | < 0.10 | 0.10 – 0.25 | > 0.25: the mix of risks has changed; review with A/E |
| A/E per quarter | 0.9 – 1.1 | 0.8 – 0.9 or 1.1 – 1.2 | outside 0.8 – 1.2 for **two quarters in a row**: recalibrate (refit the intercept on recent data) |
| Log-loss skill per quarter | > 0 | — | below 0 for **two quarters in a row**: the model is worse than the base rate; retrain |

**Claims need a different reading.** A single storm quarter (for example Ian in 2022Q3) pushes the claim A/E far above 1 with nothing wrong in the model: the model predicts the *average* year, and storms are not average. For the claim model we therefore read the A/E over **four rolling quarters** and only act on a persistent trend; for lapse, which moves slowly, the quarterly rule applies as written.

In [ ]:
# Step 9.2 A/E per quarter with the alert bands, and the status of every quarter under the rules
def status(r):
    flags = []
    if r["max_feature_psi"] > 0.25: flags.append("feature PSI > 0.25")
    if r["score_psi"] > 0.25: flags.append("score PSI > 0.25")
    elif r["score_psi"] > 0.10: flags.append("score PSI watch")
    if not 0.8 <= r["a_to_e_used"] <= 1.2: flags.append("A/E outside 0.8-1.2")
    elif not 0.9 <= r["a_to_e_used"] <= 1.1: flags.append("A/E watch")
    if r["log_loss_skill"] < 0: flags.append("skill < 0")
    return "; ".join(flags) or "ok"

parts = []
for kind in TARGETS:
    m = monitor[monitor["target"] == kind].copy()
    m["expected"] = m["predicted_rate"] * m["rows"]; m["actual"] = m["actual_rate"] * m["rows"]
    m["a_to_e_4q"] = m["actual"].rolling(4, min_periods=1).sum() / m["expected"].rolling(4, min_periods=1).sum()
    m["a_to_e_used"] = m["a_to_e_4q"] if kind == "claim" else m["a_to_e"]          # claims: 4 rolling quarters
    m["status"] = m.apply(status, axis=1)
    out_of_band = ~m["a_to_e_used"].between(0.8, 1.2)
    m["action"] = np.where(out_of_band & out_of_band.shift(fill_value=False), "recalibrate", "")
    m.loc[(m["log_loss_skill"] < 0) & (m["log_loss_skill"].shift() < 0), "action"] = "retrain"
    parts.append(m)
alerts = pd.concat(parts, ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for ax, kind in zip(axes, TARGETS):
    m = alerts[alerts["target"] == kind]
    x = np.arange(len(m))
    ax.axhspan(0.8, 1.2, color="#eef3f9", zorder=0); ax.axhspan(0.9, 1.1, color="#dbe7f4", zorder=0)
    ax.axhline(1, color="#555555", linewidth=1)
    ax.plot(x, m["a_to_e"], color="#9a9a9a", linewidth=1.5, marker="o", markersize=5, label="A/E per quarter")
    if kind == "claim":
        ax.plot(x, m["a_to_e_4q"], color="#2a6fb0", linewidth=2, label="A/E, 4 rolling quarters (used)")
    ax.set_xticks(x, m["quarter"], rotation=90, fontsize=8)
    ax.set(title=f"{kind.capitalize()}: actual / expected per quarter ({PRIMARY[kind]})", ylabel="A/E")
    ax.legend(frameon=False, fontsize=8)
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", color="#e6e6e6"); ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(REPORTS / f"fig_{OUT}_monitoring.png", dpi=150)
plt.show()
alerts[["target", "quarter", "score_psi", "max_feature_psi", "a_to_e", "a_to_e_4q", "log_loss_skill", "status", "action"]].style.format(
    {"score_psi": "{:.3f}", "max_feature_psi": "{:.3f}", "a_to_e": "{:.3f}", "a_to_e_4q": "{:.3f}", "log_loss_skill": "{:+.2%}"})

### Section 10. Appendix: extra checks
Two checks that go beyond the measures above. They do not change the results; they tell us how much to trust them.

#### 10.1 Bootstrap by policy: is the gain over the benchmarks real?
A log-loss gain of 1–3% over the segment rule could be noise. We put a 95% interval on the **difference** in log loss between the primary model and each other model, on the same rows (a paired comparison):
$$\Delta = \overline{\ell}_{\text{other}} - \overline{\ell}_{\text{primary}}, \qquad \ell_i = -\big[y_i\log p_i + (1-y_i)\log(1-p_i)\big]$$
$\Delta > 0$ means the primary model is better. Policy-years of the same policy are not independent (the same house, the same owner), so we **resample policies, not rows**: each bootstrap sample gives every policy a random weight $w_k \sim \text{Poisson}(1)$ (the Poisson bootstrap, a fast equivalent of drawing policies with replacement), and
$$\Delta^{*} = \frac{\sum_k w_k\, D_k}{\sum_k w_k\, n_k}$$
where $D_k$ is the sum of the differences over policy $k$'s rows and $n_k$ its number of rows. The 2.5th and 97.5th percentiles of $\Delta^*$ over 200 samples give the interval. **An interval that does not include 0 means the gain is real for this period**; storms in another period can still change it (Section 7).

In [ ]:
# Step 10.1 Paired bootstrap of the log-loss difference, resampling policies (Poisson bootstrap)
def row_log_loss(y, p):
    p = np.clip(np.asarray(p, dtype="float64"), 1e-7, 1 - 1e-7)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))

boot_rows = []
rng = np.random.default_rng(SEED)
for kind in TARGETS:
    p = pred[kind]
    y = p["y"].to_numpy()
    primary_col = f"p_{PRIMARY[kind]}"
    others = {n: c for n, c in MODELS[kind].items() if c != primary_col}
    ll_primary = row_log_loss(y, p[primary_col])
    diffs = pd.DataFrame({n: row_log_loss(y, p[c]) - ll_primary for n, c in others.items()})
    diffs["n"] = 1
    by_policy = diffs.groupby(p["policy_key"].to_numpy()).sum()        # D_k and n_k per policy
    D, n = by_policy[list(others)].to_numpy(), by_policy["n"].to_numpy()
    stats = np.empty((N_BOOT, len(others)))
    for b in range(N_BOOT):
        w = rng.poisson(1.0, len(n))
        stats[b] = (w @ D) / (w @ n)
    base_ll = results[kind].query("scope == @EVAL_NAME and model == 'base rate'")["log_loss"].iloc[0]
    for j, name in enumerate(others):
        est = D[:, j].sum() / n.sum()
        lo, hi = np.percentile(stats[:, j], [2.5, 97.5])
        boot_rows.append({"target": kind, "primary": PRIMARY[kind], "compared_with": name, "policies": len(n),
                          "difference": est, "ci_low": lo, "ci_high": hi,
                          "difference_pct_of_base": est / base_ll, "primary_better": "yes" if lo > 0 else ("no" if hi < 0 else "not sure")})
bootstrap = pd.DataFrame(boot_rows)
bootstrap.style.format({"policies": "{:,}", "difference": "{:+.6f}", "ci_low": "{:+.6f}", "ci_high": "{:+.6f}",
                        "difference_pct_of_base": "{:+.2%}"})

#### 10.2 Sensitivity to policy history and linkage
FEMA's data have no policy number, so notebook 03 rebuilt each policy's history with a policy key. The obvious linkage errors were removed before modeling:

- **Key collisions** (5.2% of policy-years, where one key covered more than one house) were excluded from both populations in notebook 04, so a clean-vs-collided comparison is not possible on the modeled rows.
- **Claims matched to more than one term** are shown below for the evaluation years.

What remains uncertain is the history of the policies we kept. A lapse label and the history features (tenure, premium change, earlier claims) depend on finding the previous and the next term, so we check the models on three kinds of history:

| History | Meaning | Why it matters |
|---|---|---|
| Continuous renewal | the previous term ended right before this one | the most reliable link |
| Returned after a gap | the key came back after a lapse | the link across the gap is less certain |
| First term on the key | no earlier term found | new business, or a link that was missed |

If the A/E or the skill is much worse for "returned after a gap" or "first term on the key", part of the error comes from linkage rather than from the model.

In [ ]:
# Step 10.2 Claim-match quality in the evaluation years, and the models by policy history
match = con.sql(f"""
    SELECT c.match_status, COUNT(*) AS claims
    FROM read_parquet('{PROCESSED / 'claim_links' / '*.parquet'}') c
    WHERE c.loss_year BETWEEN {2021 if EVAL_SPLIT == 'test' else 2023} AND {LAST_YEAR}
    GROUP BY ALL ORDER BY claims DESC""").df()
match["share"] = match["claims"] / match["claims"].sum()
print("claims of the evaluation years by match status (notebook 03)")
display(match.style.format({"claims": "{:,}", "share": "{:.2%}"}))

history = {}
for kind in TARGETS:
    history[kind] = segment_table(pred[kind], MODELS[kind], lambda f: f["history"])
    print(f"{kind.upper()}: by policy history")
    display(history[kind][["segment", "model", "rows", "events", "actual_rate", "a_to_e", "log_loss", "log_loss_skill"]].style.format(FMT))

### Section 11. Conclusions and save

#### 11.1 Our findings
*To be written after the final run (`EVAL_SPLIT = "valid"`). For each target:*
1. **Benchmark (Section 3):** does the primary model beat the base rate and the segment rule on log loss, Brier and ROC-AUC, overall and in each year? Is the gain over the segment rule real (Section 10.1)?
2. **Classification view (Section 4):** precision, recall and F1 at each cut-off rule; how many events the flagged group catches, and how this compares with OOT1.
3. **Calibration (Section 5):** A/E, ECE and slope: can the probabilities be used as they are, or do they need recalibration?
4. **Segments and history (Sections 6 and 10.2):** where is the model too high or too low, and is any of it linkage rather than the model?
5. **Stability and monitoring (Sections 7 and 9):** do the models drift from the development years to the out-of-time years, and which alert rules would have fired?
6. **Cohort survival (Section 8):** does the lapse model's predicted retention follow the actual retention of each new-business cohort, year after year, and where does the gap open?
7. **Recommendation:** which model to use for each target, and what to do before it goes into production (recalibration, a refit on 2014–2022, monitoring rules).

#### 11.2 Save the evaluation results
Every table goes to `reports/` with the prefix `eval_test_` (dry run) or `eval_valid_` (final run), next to the charts saved above. The row-level predictions go to `data/processed/predictions/`, so notebook 09 can explain the same rows without scoring them again.

In [ ]:
# Step 11.2 Save the evaluation tables and the row-level predictions
for kind in TARGETS:
    keep = ["period_id", "period_year", "state", "zone_group", "storm_in_term", "history", "y", *MODELS[kind].values()]
    pred[kind][keep].to_parquet(PRED_DIR / f"{OUT}_{kind}_predictions.parquet", index=False)
    results[kind].to_csv(REPORTS / f"{OUT}_{kind}_benchmark.csv", index=False)
    cm_table[cm_table["target"] == kind].to_csv(REPORTS / f"{OUT}_{kind}_confusion.csv", index=False)
    expected[kind].to_csv(REPORTS / f"{OUT}_{kind}_expected_vs_actual.csv")
    history[kind].to_csv(REPORTS / f"{OUT}_{kind}_by_history.csv", index=False)
    pd.concat({label: t for (k, label), t in seg_results.items() if k == kind}, names=["segment_type"]).to_csv(
        REPORTS / f"{OUT}_{kind}_by_segment.csv")
    pd.concat({name: deciles[(kind, name)] for name in MODELS[kind] if name != "base rate"}, names=["model", "decile"]).to_csv(
        REPORTS / f"{OUT}_{kind}_calibration_deciles.csv")
    psi_tables[kind].to_csv(REPORTS / f"{OUT}_{kind}_feature_psi.csv")
bootstrap.to_csv(REPORTS / f"{OUT}_bootstrap.csv", index=False)
cohort_tables.to_csv(REPORTS / f"{OUT}_lapse_cohort_life_tables.csv", index=False)
cohort_check.to_csv(REPORTS / f"{OUT}_lapse_cohort_check.csv", index=False)
calib.to_csv(REPORTS / f"{OUT}_calibration_summary.csv", index=False)
stability.to_csv(REPORTS / f"{OUT}_stability_by_year.csv", index=False)
alerts.to_csv(REPORTS / f"{OUT}_monitoring.csv", index=False)

run = {"evaluated": EVAL_NAME, "eval_split": EVAL_SPLIT, "last_year_read": LAST_YEAR,
       "saved_at": pd.Timestamp.now().isoformat(timespec="seconds"), "primary": PRIMARY,
       "models": {k: list(v) for k, v in registry.items()}, "base_rate_train": base_rate,
       "settings": {"n_boot": N_BOOT, "stability_rows": STABILITY_ROWS, "reference_rows": REFERENCE_ROWS,
                    "quarter_rows": QUARTER_ROWS, "seed": SEED}}
(REPORTS / f"{OUT}_run.json").write_text(json.dumps(run, indent=2, default=float))
print(f"saved {OUT}_* tables and charts to {REPORTS} and the predictions to {PRED_DIR}")